In [5]:
import sys
from pathlib import Path
sys.path.append("..")
import pandas as pd, numpy as np, lightgbm as lgb, joblib
from src.features import FEATURES
final = pd.read_parquet("data/processed/features_final.parquet")
known = final[(final.Open == 1) & (final.Sales > 0)]
# number of trees: take the best iteration from notebook 04 and add ~10 percent
N_TREES = 1500 # replace with your own best_iteration_ * 1.1
model = lgb.LGBMRegressor(n_estimators=N_TREES, learning_rate=0.03, num_leaves=63,
subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
min_child_samples=40, random_state=42, n_jobs=-1)
model.fit(known[FEATURES], known["LogSales"])
model_path = Path("models/lgbm_final.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3441
[LightGBM] [Info] Number of data points in the train set: 844338, number of used features: 26
[LightGBM] [Info] Start training from score 8.757737


['models/lgbm_final.pkl']

In [6]:
fut = final[final.Date >= "2015-08-01"].copy()
fut["Forecast"] = np.where(fut.Open == 1,
np.expm1(model.predict(fut[FEATURES])), 0)
print(fut.Forecast.describe())
print("Days:", fut.Date.nunique(), " Stores:", fut.Store.nunique())

count    41088.000000
mean      5692.143793
std       3434.397661
min          0.000000
25%       4047.420998
50%       5685.148317
75%       7525.672096
max      28973.670862
Name: Forecast, dtype: float64
Days: 48  Stores: 856


In [9]:
test = pd.read_csv("data/raw/test.csv", parse_dates=["Date"])
sub = test[["Id", "Store", "Date"]].merge(fut[["Store", "Date", "Forecast"]],
on=["Store", "Date"], how="left")
sub[["Id", "Forecast"]].rename(columns={"Forecast": "Sales"}).to_csv(
"reports/submission.csv", index=False)

In [ ]:
fut_fc = fut[["Store", "Date", "Forecast"]]
fc = fut_fc.copy()
train = pd.read_csv("data/raw/train.csv", parse_dates=["Date"], low_memory=False)
hist = train[train.Date >= "2015-05-01"][["Store", "Date", "Sales"]]
app = hist.merge(fc, on=["Store", "Date"], how="outer").sort_values(["Store", "Date"])
app["Forecast"] = app["Forecast"].round(0)
from pathlib import Path
from src.features import FEATURES
app_path = Path("app/app_data.csv")
app_path.parent.mkdir(parents=True, exist_ok=True)
app.to_csv(app_path, index=False)
feature_path = Path("app/forecast_features.parquet")
feature_rows = fut[["Date", "Open", *FEATURES]]
feature_rows.to_parquet(feature_path, index=False)
print(app.shape, app.Date.min(), app.Date.max())
print("Saved", len(feature_rows), "forecast feature rows to", feature_path)